# mini-vLLM prototype on Colab

Runtime → Change runtime type → **GPU** (A100 if available). Then run the cells in order.

Step 1 gets the code: either clone your private repo (needs a GitHub token) **or** upload `PagedAttention.zip` (zip the project folder on your laptop) with the file panel on the left.

In [ ]:
# Option A: clone (replace USER/REPO; for a private repo use https://<token>@github.com/USER/REPO.git)
!git clone https://github.com/tirthb101/PagedAttention PagedAttention

# Option B: upload PagedAttention.zip to /content, then:
# !unzip -q -o PagedAttention.zip -d /content/ && ls /content/PagedAttention

In [ ]:
%cd /content/PagedAttention
!pip -q install transformers pytest
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Correctness tests (CPU + GPU-independent, ~15 s)

In [ ]:
!python -m pytest -q

## Real model demo

OPT-125m: 6 prompts, paged KV cache, checked token-by-token against HuggingFace.

In [ ]:
!python examples/demo.py --model facebook/opt-125m --max-tokens 32

## Force preemption with a tiny KV pool

8 blocks × 16 tokens can't hold all 6 requests at once, so the scheduler must preempt (recompute or swap) — the output must still match.

In [ ]:
!python examples/demo.py --model facebook/opt-125m --kv-blocks 8 --preemption recompute
!python examples/demo.py --model facebook/opt-125m --kv-blocks 8 --preemption swap

## Parallel sampling (prompt blocks shared, copy-on-write)

In [ ]:
!python examples/demo.py --model facebook/opt-125m --n 3 --temperature 0.8 --no-check

## Bigger model (optional, A100)

In [ ]:
!python examples/demo.py --model facebook/opt-1.3b --max-tokens 64